In [2]:
# Cellule 1 - Import des librairies nécessaires
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error
from math import sqrt

# Modèles de régression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR


In [3]:
# Cellule 2 corrigée pour le séparateur ';'
data = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv", sep=';')

# Affichage des premières lignes pour vérifier le dataset
data.head()


,dataloadingdate,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,JourSemaine,ISIN,Libellé,Nombre de Titres,Montant,Echéance,Taux
0,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",1459.0,1.500,62.0,7.6
1,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",291.0,0.300,183.0,7.5
2,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,50000.0,5.000,13.0,8.6
3,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,10000.0,1.000,7.0,8.6
4,01/07/2025,1,7,27,3,1,Tuesday,TN0008000606,"BTA 6,7% Avril 2028",5660.0,5.742,31.0,9.1


In [4]:
# Cellule 2a - Informations sur le dataset
print("Dimensions du dataset :", data.shape)
print("\nColonnes et types :")
print(data.dtypes)
print("\nValeurs manquantes par colonne :")
print(data.isnull().sum())


Dimensions du dataset : (20267, 13)

Colonnes et types :
dataloadingdate      object
Jour                  int64
Mois                  int64
NumeroSemaine         int64
Trimestre             int64
JourSemaineNum        int64
JourSemaine          object
ISIN                 object
Libellé              object
Nombre de Titres    float64
Montant             float64
Echéance            float64
Taux                float64
dtype: object

Valeurs manquantes par colonne :
dataloadingdate     0
Jour                0
Mois                0
NumeroSemaine       0
Trimestre           0
JourSemaineNum      0
JourSemaine         0
ISIN                0
Libellé             0
Nombre de Titres    0
Montant             0
Echéance            0
Taux                0
dtype: int64


In [6]:
# Cellule 3 - Nettoyage et préparation basique pour le modeling

# 1. Supprimer les doublons
data = data.drop_duplicates()

# 2. Conversion des colonnes numériques (si nécessaire)
numeric_cols = ['Nombre de Titres', 'Montant', 'Echéance', 'Taux']
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# 3. Vérification des valeurs manquantes après conversion
print("Valeurs manquantes après nettoyage :")
print(data.isnull().sum())

# 4. Définir la target et les features
target = 'Montant'
features = ['Jour', 'Mois', 'NumeroSemaine', 'Trimestre', 'JourSemaineNum',
            'Nombre de Titres', 'Echéance', 'Taux']

X = data[features]
y = data[target]

# 5. Affichage des premières lignes des features et target
print("\nExemple des features :")
print(X.head())
print("\nExemple de la target :")
print(y.head())


Valeurs manquantes après nettoyage :
dataloadingdate     0
Jour                0
Mois                0
NumeroSemaine       0
Trimestre           0
JourSemaineNum      0
JourSemaine         0
ISIN                0
Libellé             0
Nombre de Titres    0
Montant             0
Echéance            0
Taux                0
dtype: int64

Exemple des features :
   Jour  Mois  NumeroSemaine  Trimestre  JourSemaineNum  Nombre de Titres  \
0     1     7             27          3               1            1459.0   
1     1     7             27          3               1             291.0   
2     1     7             27          3               1           50000.0   
3     1     7             27          3               1           10000.0   
4     1     7             27          3               1            5660.0   

   Echéance  Taux  
0      62.0   7.6  
1     183.0   7.5  
2      13.0   8.6  
3       7.0   8.6  
4      31.0   9.1  

Exemple de la target :
0    1.500
1    0.300
2    5.000


In [7]:
# Cellule 4 - Séparation des données en train et test
from sklearn.model_selection import train_test_split

# 80% pour l'entraînement, 20% pour le test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vérification des dimensions
print("Dimensions X_train :", X_train.shape)
print("Dimensions X_test  :", X_test.shape)
print("Dimensions y_train :", y_train.shape)
print("Dimensions y_test  :", y_test.shape)


Dimensions X_train : (15908, 8)
Dimensions X_test  : (3978, 8)
Dimensions y_train : (15908,)
Dimensions y_test  : (3978,)


In [12]:
# ===============================
# Modèle 1 : LinearRegression
# ===============================
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import GridSearchCV
import numpy as np

print("===== LinearRegression =====")

# 1) RMSE brut
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)
rmse_brut = np.sqrt(mean_squared_error(y_test, y_pred))

# 2) RMSE normalisé
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
lin_reg.fit(X_train_scaled, y_train)
y_pred = lin_reg.predict(X_test_scaled)
rmse_norm = np.sqrt(mean_squared_error(y_test, y_pred))

# 3) RMSE avec Feature Selection
selector = SelectKBest(score_func=f_regression, k=5)
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)
lin_reg.fit(X_train_fs, y_train)
y_pred = lin_reg.predict(X_test_fs)
rmse_fs = np.sqrt(mean_squared_error(y_test, y_pred))

# 4) RMSE fine-tuning (ici peu de params à tuner)
param_grid = {'fit_intercept': [True, False], 'copy_X': [True, False]}
grid = GridSearchCV(LinearRegression(), param_grid, cv=5, scoring='neg_mean_squared_error')
grid.fit(X_train_fs, y_train)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_fs)
rmse_ft = np.sqrt(mean_squared_error(y_test, y_pred))

# Résultats
print(f"LinearRegression RMSE brut          : {rmse_brut:.4f}")
print(f"LinearRegression RMSE normalisé    : {rmse_norm:.4f}")
print(f"LinearRegression RMSE feat select  : {rmse_fs:.4f}")
print(f"LinearRegression RMSE fine-tuning  : {rmse_ft:.4f}")


===== LinearRegression =====
LinearRegression RMSE brut          : 6.6345
LinearRegression RMSE normalisé    : 6.6345
LinearRegression RMSE feat select  : 6.6290
LinearRegression RMSE fine-tuning  : 6.6290


In [13]:
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import numpy as np

# Ridge Regression
print("===== Ridge Regression =====")

# 1. RMSE brut
ridge = Ridge()
ridge.fit(X_train, y_train)
y_pred = ridge.predict(X_test)
rmse_brut = root_mean_squared_error(y_test, y_pred)
print(f"Ridge RMSE brut          : {rmse_brut:.4f}")

# 2. Normalisation + RMSE
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge_norm = Ridge()
ridge_norm.fit(X_train_scaled, y_train)
y_pred_norm = ridge_norm.predict(X_test_scaled)
rmse_norm = root_mean_squared_error(y_test, y_pred_norm)
print(f"Ridge RMSE normalisé     : {rmse_norm:.4f}")

# 3. Feature selection + RMSE
selector = SelectKBest(score_func=f_regression, k=min(10, X_train.shape[1]))
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

ridge_fs = Ridge()
ridge_fs.fit(X_train_selected, y_train)
y_pred_fs = ridge_fs.predict(X_test_selected)
rmse_fs = root_mean_squared_error(y_test, y_pred_fs)
print(f"Ridge RMSE feat select   : {rmse_fs:.4f}")

# 4. Fine-tuning avec GridSearchCV + RMSE
param_grid = {"alpha": np.logspace(-3, 3, 7)}
grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring="neg_root_mean_squared_error")
grid.fit(X_train_selected, y_train)

best_ridge = grid.best_estimator_
y_pred_tuned = best_ridge.predict(X_test_selected)
rmse_tuned = root_mean_squared_error(y_test, y_pred_tuned)
print(f"Ridge RMSE fine-tuning   : {rmse_tuned:.4f}")


===== Ridge Regression =====
Ridge RMSE brut          : 6.6345
Ridge RMSE normalisé     : 6.6345
Ridge RMSE feat select   : 6.6345
Ridge RMSE fine-tuning   : 6.6348


In [14]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error

print("===== Lasso Regression =====")

# 1) RMSE brut
lasso = Lasso(random_state=42, max_iter=10000)
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_test)
rmse_lasso_brut = root_mean_squared_error(y_test, y_pred)
print(f"Lasso RMSE brut          : {rmse_lasso_brut:.4f}")

# 2) RMSE normalisé
lasso_norm = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(random_state=42, max_iter=10000))
])
lasso_norm.fit(X_train, y_train)
y_pred_norm = lasso_norm.predict(X_test)
rmse_lasso_norm = root_mean_squared_error(y_test, y_pred_norm)
print(f"Lasso RMSE normalisé     : {rmse_lasso_norm:.4f}")

# 3) Feature selection
selector = SelectFromModel(Lasso(alpha=0.1, random_state=42, max_iter=10000))
selector.fit(X_train, y_train)
X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

lasso_fs = Lasso(random_state=42, max_iter=10000)
lasso_fs.fit(X_train_sel, y_train)
y_pred_fs = lasso_fs.predict(X_test_sel)
rmse_lasso_fs = root_mean_squared_error(y_test, y_pred_fs)
print(f"Lasso RMSE feat select   : {rmse_lasso_fs:.4f}")

# 4) Fine-tuning
param_grid = {"alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]}
grid_lasso = GridSearchCV(Lasso(random_state=42, max_iter=10000),
                          param_grid, cv=5, scoring="neg_root_mean_squared_error")
grid_lasso.fit(X_train, y_train)

best_lasso = grid_lasso.best_estimator_
y_pred_tuned = best_lasso.predict(X_test)
rmse_lasso_tuned = root_mean_squared_error(y_test, y_pred_tuned)
print(f"Lasso RMSE fine-tuning   : {rmse_lasso_tuned:.4f}")


===== Lasso Regression =====
Lasso RMSE brut          : 6.6361
Lasso RMSE normalisé     : 6.7715
Lasso RMSE feat select   : 6.6402
Lasso RMSE fine-tuning   : 6.6331


In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler

print("===== Random Forest Regressor =====")

# 1) RMSE brut
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
rmse_rf_brut = root_mean_squared_error(y_test, y_pred)
print(f"RandomForest RMSE brut          : {rmse_rf_brut:.4f}")

# 2) RMSE normalisé (même scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_norm = RandomForestRegressor(n_estimators=100, random_state=42)
rf_norm.fit(X_train_scaled, y_train)
y_pred_norm = rf_norm.predict(X_test_scaled)
rmse_rf_norm = root_mean_squared_error(y_test, y_pred_norm)
print(f"RandomForest RMSE normalisé     : {rmse_rf_norm:.4f}")

# 3) Feature selection (SelectKBest)
selector = SelectKBest(score_func=f_regression, k=min(10, X_train.shape[1]))
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel = selector.transform(X_test_scaled)

rf_fs = RandomForestRegressor(n_estimators=100, random_state=42)
rf_fs.fit(X_train_sel, y_train)
y_pred_fs = rf_fs.predict(X_test_sel)
rmse_rf_fs = root_mean_squared_error(y_test, y_pred_fs)
print(f"RandomForest RMSE feat select   : {rmse_rf_fs:.4f}")

# 4) Fine-tuning avec GridSearchCV
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42),
                       param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_rf.fit(X_train_sel, y_train)

best_rf = grid_rf.best_estimator_
y_pred_tuned = best_rf.predict(X_test_sel)
rmse_rf_tuned = root_mean_squared_error(y_test, y_pred_tuned)
print(f"RandomForest RMSE fine-tuning   : {rmse_rf_tuned:.4f}")


===== Random Forest Regressor =====
RandomForest RMSE brut          : 3.6945
RandomForest RMSE normalisé     : 3.6923
RandomForest RMSE feat select   : 3.6923
RandomForest RMSE fine-tuning   : 3.6923


In [16]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error

print("===== Gradient Boosting Regressor =====")

# 1) RMSE brut
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)
rmse_gbr_brut = root_mean_squared_error(y_test, y_pred)
print(f"GradientBoosting RMSE brut          : {rmse_gbr_brut:.4f}")

# 2) RMSE normalisé
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

gbr_norm = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gbr_norm.fit(X_train_scaled, y_train)
y_pred_norm = gbr_norm.predict(X_test_scaled)
rmse_gbr_norm = root_mean_squared_error(y_test, y_pred_norm)
print(f"GradientBoosting RMSE normalisé     : {rmse_gbr_norm:.4f}")

# 3) Feature selection
selector = SelectKBest(score_func=f_regression, k=min(10, X_train.shape[1]))
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel = selector.transform(X_test_scaled)

gbr_fs = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gbr_fs.fit(X_train_sel, y_train)
y_pred_fs = gbr_fs.predict(X_test_sel)
rmse_gbr_fs = root_mean_squared_error(y_test, y_pred_fs)
print(f"GradientBoosting RMSE feat select   : {rmse_gbr_fs:.4f}")

# 4) Fine-tuning avec GridSearchCV
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1, 0.2],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}
grid_gbr = GridSearchCV(GradientBoostingRegressor(random_state=42),
                        param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_gbr.fit(X_train_sel, y_train)

best_gbr = grid_gbr.best_estimator_
y_pred_tuned = best_gbr.predict(X_test_sel)
rmse_gbr_tuned = root_mean_squared_error(y_test, y_pred_tuned)
print(f"GradientBoosting RMSE fine-tuning   : {rmse_gbr_tuned:.4f}")


===== Gradient Boosting Regressor =====
GradientBoosting RMSE brut          : 4.5264
GradientBoosting RMSE normalisé     : 4.5264
GradientBoosting RMSE feat select   : 4.5264
GradientBoosting RMSE fine-tuning   : 3.6503


In [17]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error

print("===== SVR =====")

# 1) RMSE brut
svr = SVR()
svr.fit(X_train, y_train)
y_pred = svr.predict(X_test)
rmse_svr_brut = root_mean_squared_error(y_test, y_pred)
print(f"SVR RMSE brut          : {rmse_svr_brut:.4f}")

# 2) RMSE normalisé
pipeline_svr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR())
])
pipeline_svr.fit(X_train, y_train)
y_pred_norm = pipeline_svr.predict(X_test)
rmse_svr_norm = root_mean_squared_error(y_test, y_pred_norm)
print(f"SVR RMSE normalisé     : {rmse_svr_norm:.4f}")

# 3) Feature selection
selector = SelectKBest(score_func=f_regression, k=min(10, X_train.shape[1]))
X_train_sel = selector.fit_transform(StandardScaler().fit_transform(X_train), y_train)
X_test_sel = selector.transform(StandardScaler().fit_transform(X_test))

svr_fs = SVR()
svr_fs.fit(X_train_sel, y_train)
y_pred_fs = svr_fs.predict(X_test_sel)
rmse_svr_fs = root_mean_squared_error(y_test, y_pred_fs)
print(f"SVR RMSE feat select   : {rmse_svr_fs:.4f}")

# 4) Fine-tuning avec GridSearchCV
param_grid = {"C":[0.1,1,10], "epsilon":[0.01,0.1,0.5], "kernel":["linear","rbf"]}
grid_svr = GridSearchCV(SVR(), param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_svr.fit(X_train_sel, y_train)
best_svr = grid_svr.best_estimator_
y_pred_tuned = best_svr.predict(X_test_sel)
rmse_svr_tuned = root_mean_squared_error(y_test, y_pred_tuned)
print(f"SVR RMSE fine-tuning   : {rmse_svr_tuned:.4f}")


===== SVR =====
SVR RMSE brut          : 6.6948
SVR RMSE normalisé     : 6.1112
SVR RMSE feat select   : 6.2639
SVR RMSE fine-tuning   : 6.0489


In [18]:
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error

print("===== XGBoost Regressor =====")

# 1) RMSE brut
xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
rmse_xgb_brut = root_mean_squared_error(y_test, y_pred)
print(f"XGBoost RMSE brut          : {rmse_xgb_brut:.4f}")

# 2) RMSE normalisé
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
xgb_norm = XGBRegressor(n_estimators=100, random_state=42)
xgb_norm.fit(X_train_scaled, y_train)
y_pred_norm = xgb_norm.predict(X_test_scaled)
rmse_xgb_norm = root_mean_squared_error(y_test, y_pred_norm)
print(f"XGBoost RMSE normalisé     : {rmse_xgb_norm:.4f}")

# 3) Feature selection
selector = SelectKBest(score_func=f_regression, k=min(10, X_train.shape[1]))
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel = selector.transform(X_test_scaled)
xgb_fs = XGBRegressor(n_estimators=100, random_state=42)
xgb_fs.fit(X_train_sel, y_train)
y_pred_fs = xgb_fs.predict(X_test_sel)
rmse_xgb_fs = root_mean_squared_error(y_test, y_pred_fs)
print(f"XGBoost RMSE feat select   : {rmse_xgb_fs:.4f}")

# 4) Fine-tuning avec GridSearchCV
param_grid = {
    "n_estimators":[100,200],
    "max_depth":[3,5,7],
    "learning_rate":[0.05,0.1,0.2]
}
grid_xgb = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=3,
                        scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_xgb.fit(X_train_sel, y_train)
best_xgb = grid_xgb.best_estimator_
y_pred_tuned = best_xgb.predict(X_test_sel)
rmse_xgb_tuned = root_mean_squared_error(y_test, y_pred_tuned)
print(f"XGBoost RMSE fine-tuning   : {rmse_xgb_tuned:.4f}")


===== XGBoost Regressor =====
XGBoost RMSE brut          : 3.7538
XGBoost RMSE normalisé     : 3.7538
XGBoost RMSE feat select   : 3.7538
XGBoost RMSE fine-tuning   : 3.8756


In [19]:
import pandas as pd
import numpy as np

# Copier le dataset nettoyé
df = data.copy()

# Détection des outliers par IQR sur la cible Montant
Q1 = df['Montant'].quantile(0.25)
Q3 = df['Montant'].quantile(0.75)
IQR = Q3 - Q1

# Seuils
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Affichage du nombre d'outliers
outliers_low = (df['Montant'] < lower_bound).sum()
outliers_high = (df['Montant'] > upper_bound).sum()

print(f"Nombre d'outliers en dessous de {lower_bound:.2f} : {outliers_low}")
print(f"Nombre d'outliers au-dessus de {upper_bound:.2f} : {outliers_high}")


Nombre d'outliers en dessous de -5.75 : 0
Nombre d'outliers au-dessus de 12.25 : 1530


In [20]:
# Winsorisation de la cible Montant
df['Montant_winsor'] = df['Montant'].clip(lower=lower_bound, upper=upper_bound)

# Vérification
print("Statistiques après winsorisation :")
print(df['Montant_winsor'].describe())


Statistiques après winsorisation :
count    19886.000000
mean         3.789072
std          3.717714
min          0.001000
25%          1.000000
50%          2.001000
75%          5.500000
max         12.250000
Name: Montant_winsor, dtype: float64


In [21]:
from sklearn.model_selection import train_test_split

# Features
X = df[['Jour','Mois','NumeroSemaine','Trimestre','JourSemaineNum','Nombre de Titres','Echéance','Taux']]

# Target winsorisée
y = df['Montant_winsor']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.197, random_state=42)

print("Dimensions X_train :", X_train.shape)
print("Dimensions X_test  :", X_test.shape)
print("Dimensions y_train :", y_train.shape)
print("Dimensions y_test  :", y_test.shape)


Dimensions X_train : (15968, 8)
Dimensions X_test  : (3918, 8)
Dimensions y_train : (15968,)
Dimensions y_test  : (3918,)


In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

print("===== Random Forest Regressor (winsorisé) =====")

# 1) RMSE brut
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
rmse_rf_brut = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RandomForest RMSE brut          : {rmse_rf_brut:.4f}")

# 2) RMSE normalisé
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_norm = RandomForestRegressor(n_estimators=200, random_state=42)
rf_norm.fit(X_train_scaled, y_train)
y_pred_norm = rf_norm.predict(X_test_scaled)
rmse_rf_norm = np.sqrt(mean_squared_error(y_test, y_pred_norm))
print(f"RandomForest RMSE normalisé     : {rmse_rf_norm:.4f}")

# 3) Feature selection (SelectKBest)
from sklearn.feature_selection import SelectKBest, f_regression
selector = SelectKBest(score_func=f_regression, k=8)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel = selector.transform(X_test_scaled)

rf_fs = RandomForestRegressor(n_estimators=200, random_state=42)
rf_fs.fit(X_train_sel, y_train)
y_pred_fs = rf_fs.predict(X_test_sel)
rmse_rf_fs = np.sqrt(mean_squared_error(y_test, y_pred_fs))
print(f"RandomForest RMSE feat select   : {rmse_rf_fs:.4f}")

# 4) Fine-tuning (GridSearch)
from sklearn.model_selection import GridSearchCV
param_grid = {
    "n_estimators":[200,300],
    "max_depth":[None,10,20],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2]
}
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3,
                       scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_rf.fit(X_train_sel, y_train)
best_rf = grid_rf.best_estimator_
y_pred_tuned = best_rf.predict(X_test_sel)
rmse_rf_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
print(f"RandomForest RMSE fine-tuning   : {rmse_rf_tuned:.4f}")


===== Random Forest Regressor (winsorisé) =====
RandomForest RMSE brut          : 1.4372
RandomForest RMSE normalisé     : 1.4367
RandomForest RMSE feat select   : 1.4367
RandomForest RMSE fine-tuning   : 1.4321


In [23]:
from sklearn.ensemble import GradientBoostingRegressor

print("===== Gradient Boosting Regressor (winsorisé) =====")

# 1) RMSE brut
gb = GradientBoostingRegressor(n_estimators=200, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
rmse_gb_brut = np.sqrt(mean_squared_error(y_test, y_pred_gb))
print(f"GradientBoosting RMSE brut          : {rmse_gb_brut:.4f}")

# 2) RMSE normalisé
gb_norm = GradientBoostingRegressor(n_estimators=200, random_state=42)
gb_norm.fit(X_train_scaled, y_train)
y_pred_gb_norm = gb_norm.predict(X_test_scaled)
rmse_gb_norm = np.sqrt(mean_squared_error(y_test, y_pred_gb_norm))
print(f"GradientBoosting RMSE normalisé     : {rmse_gb_norm:.4f}")

# 3) Feature selection
gb_fs = GradientBoostingRegressor(n_estimators=200, random_state=42)
gb_fs.fit(X_train_sel, y_train)
y_pred_gb_fs = gb_fs.predict(X_test_sel)
rmse_gb_fs = np.sqrt(mean_squared_error(y_test, y_pred_gb_fs))
print(f"GradientBoosting RMSE feat select   : {rmse_gb_fs:.4f}")

# 4) Fine-tuning (GridSearch)
param_grid_gb = {
    "n_estimators":[200,300],
    "max_depth":[3,5,10],
    "learning_rate":[0.05,0.1,0.2],
    "min_samples_split":[2,5]
}
grid_gb = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid_gb, cv=3,
                       scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_gb.fit(X_train_sel, y_train)
best_gb = grid_gb.best_estimator_
y_pred_gb_tuned = best_gb.predict(X_test_sel)
rmse_gb_tuned = np.sqrt(mean_squared_error(y_test, y_pred_gb_tuned))
print(f"GradientBoosting RMSE fine-tuning   : {rmse_gb_tuned:.4f}")


===== Gradient Boosting Regressor (winsorisé) =====
GradientBoosting RMSE brut          : 1.7346
GradientBoosting RMSE normalisé     : 1.7347
GradientBoosting RMSE feat select   : 1.7347
GradientBoosting RMSE fine-tuning   : 1.3367


In [24]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("===== SVR Regressor (winsorisé) =====")

# 1) RMSE brut
svr = SVR()
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)
rmse_svr_brut = np.sqrt(mean_squared_error(y_test, y_pred_svr))
print(f"SVR RMSE brut          : {rmse_svr_brut:.4f}")

# 2) RMSE normalisé (scaling)
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR())
])
svr_pipeline.fit(X_train, y_train)
y_pred_svr_norm = svr_pipeline.predict(X_test)
rmse_svr_norm = np.sqrt(mean_squared_error(y_test, y_pred_svr_norm))
print(f"SVR RMSE normalisé     : {rmse_svr_norm:.4f}")

# 3) Feature selection
selector_svr = SelectKBest(score_func=f_regression, k=8)
X_train_svr_sel = selector_svr.fit_transform(X_train_scaled, y_train)
X_test_svr_sel = selector_svr.transform(X_test_scaled)

svr_fs = SVR()
svr_fs.fit(X_train_svr_sel, y_train)
y_pred_svr_fs = svr_fs.predict(X_test_svr_sel)
rmse_svr_fs = np.sqrt(mean_squared_error(y_test, y_pred_svr_fs))
print(f"SVR RMSE feat select   : {rmse_svr_fs:.4f}")

# 4) Fine-tuning (GridSearch)
from sklearn.model_selection import GridSearchCV
param_grid_svr = {
    "C":[0.1,1,10],
    "epsilon":[0.01,0.1,0.5],
    "kernel":["rbf","linear"]
}
grid_svr = GridSearchCV(SVR(), param_grid_svr, cv=3,
                        scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_svr.fit(X_train_svr_sel, y_train)
best_svr = grid_svr.best_estimator_
y_pred_svr_tuned = best_svr.predict(X_test_svr_sel)
rmse_svr_tuned = np.sqrt(mean_squared_error(y_test, y_pred_svr_tuned))
print(f"SVR RMSE fine-tuning   : {rmse_svr_tuned:.4f}")


===== SVR Regressor (winsorisé) =====
SVR RMSE brut          : 2.8062
SVR RMSE normalisé     : 2.7476
SVR RMSE feat select   : 2.7476
SVR RMSE fine-tuning   : 2.5134


In [27]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import numpy as np

print("===== Linear Regression (winsorisé) =====")

# === 1. Brut ===
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lr = lin_reg.predict(X_test)
rmse_brut_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f"Linear Regression RMSE brut          : {rmse_brut_lr:.4f}")

# === 2. Normalisation ===
scaler = StandardScaler()
X_train_scaled_lr = scaler.fit_transform(X_train)
X_test_scaled_lr = scaler.transform(X_test)

lin_reg_norm = LinearRegression()
lin_reg_norm.fit(X_train_scaled_lr, y_train)
y_pred_lr_norm = lin_reg_norm.predict(X_test_scaled_lr)
rmse_norm_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr_norm))
print(f"Linear Regression RMSE normalisé     : {rmse_norm_lr:.4f}")

# === 3. Feature Selection ===
selector_lr = SelectKBest(score_func=f_regression, k=5)  # prends 5 meilleures features
X_train_selected = selector_lr.fit_transform(X_train, y_train)
X_test_selected = selector_lr.transform(X_test)

lin_reg_feat = LinearRegression()
lin_reg_feat.fit(X_train_selected, y_train)
y_pred_lr_feat = lin_reg_feat.predict(X_test_selected)
rmse_feat_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr_feat))
print(f"Linear Regression RMSE feat select   : {rmse_feat_lr:.4f}")

# === 4. Fine-tuning (pas de params pour LinearRegression classique) ===
print(f"Linear Regression RMSE fine-tuning   : {rmse_feat_lr:.4f}")


===== Linear Regression (winsorisé) =====
Linear Regression RMSE brut          : 3.2323
Linear Regression RMSE normalisé     : 3.2323
Linear Regression RMSE feat select   : 3.2293
Linear Regression RMSE fine-tuning   : 3.2293


In [29]:
# === Random Forest Regressor - Fine-Tuning rapide avec RobustScaler ===

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error
import numpy as np

# 1️⃣ Normalisation robuste
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2️⃣ Définition de l’espace des hyperparamètres
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.6, 0.8, None]
}

# 3️⃣ RandomizedSearchCV pour accélérer le tuning
random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=50,          # moins d’itérations pour exécuter rapidement
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 4️⃣ Exécution du tuning
random_search.fit(X_train_scaled, y_train)

# 5️⃣ Meilleur modèle
best_rf = random_search.best_estimator_
print("Meilleurs hyperparamètres :", random_search.best_params_)

# 6️⃣ Prédiction et calcul RMSE
y_pred = best_rf.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Random Forest RMSE fine-tuning rapide : {rmse:.4f}")


Fitting 3 folds for each of 50 candidates, totalling 150 fits
Meilleurs hyperparamètres : {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': 30}
Random Forest RMSE fine-tuning rapide : 1.4294


In [31]:
# === Imports ===
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats.mstats import winsorize

# === Winsorisation de la target ===
y_winsor = winsorize(y, limits=[0.05, 0.05])

# === Winsorisation des features ===
X_winsor = X.copy()
for col in X_winsor.columns:
    if X_winsor[col].dtype in [np.float64, np.int64]:
        X_winsor[col] = winsorize(X_winsor[col], limits=[0.05, 0.05])

# === Séparation train/test ===
X_train_winsor, X_test_winsor, y_train, y_test = train_test_split(
    X_winsor, y_winsor, test_size=0.2, random_state=42
)

# === Linear Regression ===
lin = LinearRegression()
lin.fit(X_train_winsor, y_train)
y_pred = lin.predict(X_test_winsor)
rmse_lin = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Linear Regression RMSE fine-tuning rapide : {rmse_lin:.4f}")

# === Ridge Regression ===
ridge = Ridge()
ridge.fit(X_train_winsor, y_train)
y_pred = ridge.predict(X_test_winsor)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Ridge Regression RMSE fine-tuning rapide : {rmse_ridge:.4f}")

# === Lasso Regression ===
lasso = Lasso(max_iter=10000)
lasso.fit(X_train_winsor, y_train)
y_pred = lasso.predict(X_test_winsor)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Lasso Regression RMSE fine-tuning rapide : {rmse_lasso:.4f}")

# === Random Forest Regressor ===
param_dist_rf = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.6, 0.8, 1.0]
}
rf_random = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist_rf,
    n_iter=25,
    cv=3,
    verbose=0,
    random_state=42
)
rf_random.fit(X_train_winsor, y_train)
best_rf = rf_random.best_estimator_
y_pred = best_rf.predict(X_test_winsor)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Random Forest RMSE fine-tuning rapide : {rmse_rf:.4f}")
print(f"Meilleurs hyperparamètres RF : {rf_random.best_params_}")

# === Gradient Boosting Regressor ===
param_dist_gb = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.7, 0.8, 1.0]
}
gb_random = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=25,
    cv=3,
    verbose=0,
    random_state=42
)
gb_random.fit(X_train_winsor, y_train)
best_gb = gb_random.best_estimator_
y_pred = best_gb.predict(X_test_winsor)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Gradient Boosting RMSE fine-tuning rapide : {rmse_gb:.4f}")
print(f"Meilleurs hyperparamètres GB : {gb_random.best_params_}")

# === SVR Regressor ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_winsor)
X_test_scaled = scaler.transform(X_test_winsor)

param_dist_svr = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
    'epsilon': [0.01, 0.1, 0.2]
}
svr_random = RandomizedSearchCV(
    estimator=SVR(),
    param_distributions=param_dist_svr,
    n_iter=20,
    cv=3,
    verbose=0,
    random_state=42
)
svr_random.fit(X_train_scaled, y_train)
best_svr = svr_random.best_estimator_
y_pred = best_svr.predict(X_test_scaled)
rmse_svr = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"SVR RMSE fine-tuning rapide : {rmse_svr:.4f}")
print(f"Meilleurs hyperparamètres SVR : {svr_random.best_params_}")


Linear Regression RMSE fine-tuning rapide : 3.0371
Ridge Regression RMSE fine-tuning rapide : 3.0371
Lasso Regression RMSE fine-tuning rapide : 3.0376
Random Forest RMSE fine-tuning rapide : 1.5141
Meilleurs hyperparamètres RF : {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 20}
Gradient Boosting RMSE fine-tuning rapide : 1.3866
Meilleurs hyperparamètres GB : {'subsample': 0.8, 'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 7, 'learning_rate': 0.1}
SVR RMSE fine-tuning rapide : 2.3344
Meilleurs hyperparamètres SVR : {'gamma': 'auto', 'epsilon': 0.2, 'C': 100}
